# 📦 Notebook 01: Setup e Configuração do Ambiente

**Projeto:** XAI-AHP-Gaussian ESGE Framework  
**Autor:** Cesar Yoshio Machado Pedroza  
**Instituição:** USP/Esalq - MBA Data Science and Analytics  
**Data:** 2026-04-16  

---

## 🎯 Objetivo

Este notebook realiza a configuração inicial do ambiente de pesquisa:
1. Verificação e instalação de dependências Python
2. Download de recursos NLP (NLTK, VADER)
3. Criação da estrutura de diretórios
4. Validação do ambiente

---

## 📋 Pré-requisitos

- Python 3.10+
- pip atualizado: `python -m pip install --upgrade pip`
- Conexão com internet (para download de pacotes)

---

## 📚 Referências Metodológicas

A estrutura de diretórios segue o padrão **Cookiecutter Data Science** (DrivenData, 2020):
- `data/raw/` - Dados brutos imutáveis
- `data/processed/` - Dados processados e limpos
- `outputs/` - Resultados, gráficos, tabelas
- `src/` - Código Python modular

**Referência:**  
DrivenData. (2020). Cookiecutter Data Science. Retrieved from https://drivendata.github.io/cookiecutter-data-science/

---

## 1️⃣ Importação de Bibliotecas

In [1]:
"""Setup Inicial - Importações e Configurações Básicas."""

import os
import sys
import subprocess
import logging
from pathlib import Path
from typing import List, Optional

# Configuração de logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

print("✅ Bibliotecas básicas importadas com sucesso.")

✅ Bibliotecas básicas importadas com sucesso.


## 2️⃣ Verificação e Instalação de Dependências

### 📖 Metodologia

Implementamos verificação automática de dependências para garantir **reprodutibilidade** do ambiente (Sandve et al., 2013).

O sistema verifica a disponibilidade de cada pacote e instala apenas os ausentes, evitando:
- ❌ Reinstalações desnecessárias
- ❌ Conflitos de versão
- ❌ Sobrecarga de rede

**Referência:**  
Sandve, G. K., et al. (2013). Ten simple rules for reproducible computational research. *PLOS Computational Biology*, 9(10), e1003285.

In [2]:
def check_and_install_dependencies(packages: List[str]) -> None:
    """
    Verifica a disponibilidade de pacotes Python e instala os ausentes.
    
    Parameters
    ----------
    packages : List[str]
        Lista de nomes de pacotes a serem verificados.
        
    Returns
    -------
    None
        Instalação silenciosa via pip.
        
    Examples
    --------
    >>> check_and_install_dependencies(['pandas', 'numpy', 'scipy'])
    ✅ Todas as dependências estão instaladas.
    
    Notes
    -----
    - Requer pip funcional no ambiente
    - Conexão com internet necessária para downloads
    - Timeout padrão: 120 segundos por pacote
    """
    missing = []
    
    # Verifica cada pacote
    for pkg in packages:
        try:
            __import__(pkg.replace('-', '_'))  # Normaliza nomes com hífen
        except ImportError:
            missing.append(pkg)
    
    # Instalação se necessário
    if missing:
        logger.warning(f"⚠️ Pacotes ausentes: {missing}")
        logger.info("🔧 Instalando dependências...")
        
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", *missing],
                stdout=subprocess.DEVNULL,  # Silencia output verboso
                stderr=subprocess.PIPE,
                timeout=120 * len(missing)  # 2min por pacote
            )
            logger.info(f"✅ {len(missing)} pacotes instalados com sucesso.")
        except subprocess.CalledProcessError as e:
            logger.error(f"❌ Falha na instalação: {e.stderr.decode()}")
            raise
    else:
        logger.info("✅ Todas as dependências já estão instaladas.")


# Lista de dependências críticas
REQUIRED_PACKAGES = [
    "pandas",           # Manipulação de dados
    "numpy",            # Computação numérica
    "scipy",            # Algoritmos científicos
    "scikit-learn",     # Machine Learning
    "xgboost",          # Gradient Boosting
    "shap",             # SHAP values
    "lime",             # LIME explanations
    "dice-ml",          # Counterfactuals
    "pdfplumber",       # Extração de PDFs
    "yfinance",         # Yahoo Finance API
    "nltk",             # Natural Language Toolkit
    "vaderSentiment",   # Sentiment Analysis
    "matplotlib",       # Visualizações
    "seaborn",          # Gráficos estatísticos
    "plotly",           # Gráficos interativos
    "streamlit",        # Dashboard DSS
    "openpyxl"          # Suporte Excel
]

# Executa verificação
check_and_install_dependencies(REQUIRED_PACKAGES)

2026-04-21 00:00:26 | WARNING | ⚠️ Pacotes ausentes: ['scikit-learn']
2026-04-21 00:00:26 | INFO | 🔧 Instalando dependências...
2026-04-21 00:00:29 | INFO | ✅ 1 pacotes instalados com sucesso.


## 3️⃣ Download de Recursos NLP

### 📖 Fundamento Teórico

**NLTK (Natural Language Toolkit)** é utilizado para processamento de texto extraído dos PDFs.

**VADER (Valence Aware Dictionary and sEntiment Reasoner)** (Hutto & Gilbert, 2014) é um analisador de sentimentos otimizado para texto corporativo.

**Recursos baixados:**
- `punkt` - Tokenizador de sentenças
- `punkt_tab` - Tabelas de pontuação
- `vader_lexicon` - Léxico de sentimentos
- `stopwords` - Palavras vazias (inglês/português)

**Referência:**  
Hutto, C. J., & Gilbert, E. (2014). VADER: A parsimonious rule-based model for sentiment analysis of social media text. *Proceedings of ICWSM*, 8(1), 216-225.

In [3]:
def setup_nlp_resources() -> None:
    """
    Baixa recursos NLTK necessários para análise de texto.
    
    Returns
    -------
    None
        Download silencioso dos recursos.
        
    Notes
    -----
    - Armazenamento padrão: ~/nltk_data/
    - Tamanho total: ~50MB
    - Execução idempotente (safe to run multiple times)
    """
    import nltk
    
    resources = [
        'punkt',          # Sentence tokenizer
        'punkt_tab',      # Punctuation tables
        'vader_lexicon',  # Sentiment lexicon
        'stopwords'       # Common words to filter
    ]
    
    logger.info("📥 Baixando recursos NLTK...")
    
    for resource in resources:
        try:
            nltk.download(resource, quiet=True, raise_on_error=True)
        except Exception as e:
            logger.warning(f"⚠️ Recurso '{resource}' pode já estar instalado: {e}")
    
    logger.info("✅ Recursos NLP configurados com sucesso.")


# Executa download
setup_nlp_resources()

2026-04-21 00:00:29 | INFO | 📥 Baixando recursos NLTK...
2026-04-21 00:00:29 | INFO | ✅ Recursos NLP configurados com sucesso.


## 4️⃣ Criação da Estrutura de Diretórios

### 📖 Padrão Organizacional

A estrutura segue o **Cookiecutter Data Science** com adaptações para XAI:

```
teck-esge-xai/
├── data/
│   ├── raw/          # PDFs originais (imutáveis)
│   └── processed/    # CSVs limpos e estruturados
├── outputs/
│   ├── figures/      # Gráficos (PNG, PDF)
│   ├── tables/       # Tabelas (CSV, XLSX)
│   ├── models/       # Modelos treinados (.pkl)
│   └── logs/         # Logs de execução
└── powerbi_data/     # Star Schema para BI
```

**Princípio de Imutabilidade:**  
Dados brutos (`data/raw/`) **nunca** são modificados. Todas as transformações geram novos arquivos em `data/processed/`.

In [4]:
def create_project_structure(base_dir: Optional[Path] = None) -> Path:
    """
    Cria a estrutura completa de diretórios do projeto.
    
    Parameters
    ----------
    base_dir : Optional[Path], default=None
        Diretório raiz do projeto. Se None, usa o diretório atual.
        
    Returns
    -------
    Path
        Caminho absoluto do diretório base criado.
        
    Examples
    --------
    >>> base = create_project_structure(Path("/home/user/teck-esge-xai"))
    ✅ Estrutura de diretórios criada: /home/user/teck-esge-xai
    """
    if base_dir is None:
        # Auto-detecta baseado no notebook location
        base_dir = Path.cwd()
        if "notebooks" in str(base_dir):
            base_dir = base_dir.parent
    
    # Definição da hierarquia
    directories = [
        base_dir / "data" / "raw",
        base_dir / "data" / "processed",
        base_dir / "outputs" / "figures",
        base_dir / "outputs" / "tables",
        base_dir / "outputs" / "models",
        base_dir / "outputs" / "logs",
        base_dir / "powerbi_data",
        base_dir / "src",
        base_dir / "notebooks",
        base_dir / "tests",
        base_dir / "docs"
    ]
    
    # Criação recursiva
    for directory in directories:
        directory.mkdir(parents=True, exist_ok=True)
        
        # Cria .gitkeep para versionar pastas vazias
        gitkeep = directory / ".gitkeep"
        if not gitkeep.exists() and directory.name not in ["src", "notebooks", "tests", "docs"]:
            gitkeep.touch()
    
    logger.info(f"✅ Estrutura de diretórios criada: {base_dir}")
    return base_dir


# Executa criação (modifique o caminho conforme necessário)
BASE_DIR = create_project_structure(
    Path(r"C:\Users\user\Documents\teck-esge-xai")  # Windows
    # Path("/home/user/teck-esge-xai")  # Linux/Mac
)

print(f"\n📁 Diretório Base: {BASE_DIR}")

2026-04-21 00:00:29 | INFO | ✅ Estrutura de diretórios criada: C:\Users\user\Documents\teck-esge-xai



📁 Diretório Base: C:\Users\user\Documents\teck-esge-xai


## 5️⃣ Validação do Ambiente

In [5]:
def validate_environment() -> bool:
    """
    Valida que o ambiente está configurado corretamente.
    
    Returns
    -------
    bool
        True se validação passou, False caso contrário.
    """
    print("\n" + "="*70)
    print("VALIDAÇÃO DO AMBIENTE")
    print("="*70)
    
    checks = []
    
    # 1. Versão Python
    python_version = sys.version_info
    checks.append((
        "Python Version",
        python_version >= (3, 10),
        f"{python_version.major}.{python_version.minor}.{python_version.micro}"
    ))
    
    # 2. Pacotes críticos
    critical_packages = ["pandas", "numpy", "sklearn", "shap"]
    for pkg in critical_packages:
        try:
            mod = __import__(pkg)
            version = getattr(mod, '__version__', 'unknown')
            checks.append((f"Package: {pkg}", True, version))
        except ImportError:
            checks.append((f"Package: {pkg}", False, "NOT INSTALLED"))
    
    # 3. Diretórios
    required_dirs = ["data/raw", "outputs", "powerbi_data"]
    for dir_name in required_dirs:
        dir_path = BASE_DIR / dir_name
        checks.append((f"Directory: {dir_name}", dir_path.exists(), str(dir_path)))
    
    # Exibe resultados
    all_passed = True
    for check_name, passed, detail in checks:
        status = "✅" if passed else "❌"
        print(f"{status} {check_name:.<50} {detail}")
        if not passed:
            all_passed = False
    
    print("="*70)
    if all_passed:
        print("🎉 AMBIENTE VALIDADO COM SUCESSO!")
    else:
        print("⚠️ AMBIENTE COM PROBLEMAS - VERIFIQUE OS ERROS ACIMA")
    print("="*70)
    
    return all_passed


# Executa validação
validate_environment()


VALIDAÇÃO DO AMBIENTE
✅ Python Version.................................... 3.13.9
✅ Package: pandas................................... 2.3.3
✅ Package: numpy.................................... 2.3.5
✅ Package: sklearn.................................. 1.7.2
✅ Package: shap..................................... 0.51.0
✅ Directory: data/raw............................... C:\Users\user\Documents\teck-esge-xai\data\raw
✅ Directory: outputs................................ C:\Users\user\Documents\teck-esge-xai\outputs
✅ Directory: powerbi_data........................... C:\Users\user\Documents\teck-esge-xai\powerbi_data
🎉 AMBIENTE VALIDADO COM SUCESSO!


True

## ✅ Checklist de Conclusão

Antes de prosseguir para o próximo notebook, verifique:

- [ ] Todas as dependências instaladas
- [ ] Recursos NLTK baixados
- [ ] Estrutura de diretórios criada
- [ ] Validação do ambiente passou
- [ ] Caminhos em `config.py` ajustados (se necessário)

---

## 📚 Próximos Passos

1. **Notebook 02:** Extração de dados (PDFs + API Financeira)
2. **Notebook 03:** Modelagem ML e XAI
3. **Notebook 04:** Análise financeira
4. **Notebook 05:** AHP-Gaussiano
5. **Notebook 06:** Exportação Power BI

---

## 📖 Referências Bibliográficas

- DrivenData. (2020). *Cookiecutter Data Science*. https://drivendata.github.io/cookiecutter-data-science/
- Hutto, C. J., & Gilbert, E. (2014). VADER: A parsimonious rule-based model for sentiment analysis of social media text. *Proceedings of ICWSM*, 8(1), 216-225.
- Sandve, G. K., et al. (2013). Ten simple rules for reproducible computational research. *PLOS Computational Biology*, 9(10), e1003285.

---

**Última Atualização:** 2026-04-16  
**Versão:** 1.0  
**Licença:** MIT